In [37]:
# https://pypi.org/project/geopy/
from geopy.distance import geodesic
import gpxpy

In [38]:
KM_E_PER_DAY = 25

In [56]:
# Rallye Pantin 2023 50K
# https://www.cyclo-sport-pantin.fr/nos-traces/rallye-pantin-2023-50k
# gpx_file_name = "test.gpx"
# Mountain bike loop at Middlesex Fells reservation
# https://www.topografix.com/gpx_sample_files.asp
# gpx_file_name = "fells_loop.gpx"
# gpx_file_name = "gr38.gpx"
gpx_file_name = "gr38_day_0.gpx"
gpx_file = open(gpx_file_name, "r")
gpx = gpxpy.parse(gpx_file)

In [ ]:
for track in gpx.tracks:
    for segment in track.segments:
        for point in segment.points:
            print(f'Point at ({point.latitude},{point.longitude}) -> {point.elevation}')

In [ ]:
for waypoint in gpx.waypoints:
    print(f'waypoint {waypoint.name} -> ({waypoint.latitude},{waypoint.longitude})')

In [ ]:
for route in gpx.routes:
    print('Route:')
    for point in route.points:
        print(f'Point at ({point.latitude},{point.longitude}) -> {point.elevation}')

In [41]:
def print_gpx_info(gpx, gpx_file: str) -> None:
    print(f'File: {gpx_file}')

    if gpx.name:
        print(f'  GPX name: {gpx.name}')
    if gpx.description:
        print(f'  GPX description: {gpx.description}')
    if gpx.author_name:
        print(f'  Author: {gpx.author_name}')
    if gpx.author_email:
        print(f'  Email: {gpx.author_email}')

    # print_gpx_part_info(gpx)

    for track_no, track in enumerate(gpx.tracks):
        for segment_no, segment in enumerate(track.segments):
            print(f'    Track #{track_no}, Segment #{segment_no}')
            # print_gpx_part_info(segment, indentation='        ')

print_gpx_info(gpx, gpx_file_name)

File: gr38_day_0.gpx
  GPX name: GR38
  GPX description: Un morceau du GR38, de Redon à Gourin
  Author: Anne L'Hôte
  Email: anne.lhote@gmail.com
    Track #0, Segment #0


## Distance totale

In [ ]:
# https://github.com/bhyman67/Garmin-Connect-Activities/blob/main/03%20-%20Extract%20GPX%20Activity%20Metrics.py
def get_total_distance(gpx):
    # Initialize total distance
    total_distance = 0
    total_distance_e = 0
    # for route in gpx.routes:
    #     print('Route:')
    #     for point in route.points:
    #         print(f'Point at ({point.latitude},{point.longitude}) -> {point.elevation}')
    # Iterate through track points and calculate distance
    for track in gpx.tracks:
      for segment in track.segments:
        for i in range(1, len(segment.points)):
          point1 = segment.points[i - 1]
          point2 = segment.points[i]
          elevation_gain = point2.elevation - point1.elevation
          distance = geodesic((point1.latitude, point1.longitude), (point2.latitude, point2.longitude)).km
          total_distance += distance
          if elevation_gain > 0:
            total_distance_e += distance + (elevation_gain / 1000)
          else:
            total_distance_e += distance
    return total_distance, total_distance_e

distance, distance_e = get_total_distance(gpx)
print(f"Distance: {round(distance)}km")
print(f"Distance e: {round(distance_e)}km-e")

0.014835103861301695
0.005374899287759033
0.00932093989914308
0.017812497104146747
0.011742068420860086
0.002982399949381673
0.0030884091705828806
0.011082977716446793
0.01924385444056012
0.020431635319044197
0.02218852854040594
0.01370099436676886
0.0186498551509286
0.00023471626744412706
0.0
0.00011118347943906131
7.513096664616807e-05
0.00015026193335906582
0.0004447339192004791
0.0010034678472538039
0.0009015713763026955
0.00011118348164338145
0.00013418802418702801
0.00015026189607269264
0.0008912124979290327
0.00039176302701889076
0.005940875014638038
0.0008994079147883409
0.0005026481449865608
0.001011870409664881
0.00026837610435572205
0.0007926570539378944
0.0013244162521704997
0.00022236695941798434
0.00013418804285450704
0.0001502619677772496
0.0006112448904197693
0.0011486934448530978
0.008380463580387547
0.017139489390460345
0.001423605317684206
0.0011897031997044317
0.0011897031106121144
0.001141988355615401
0.0011238273444315564
0.0011034512569129467
0.001320058974396271

## D+ & D-

In [45]:
(uphill, downhill) = gpx.get_uphill_downhill() # in meters
print(f"D+: {round(uphill)}m / D-: {round(downhill)}m")

D+: 139m / D-: 140m


## Distance en km effort

In [ ]:
distance_e2 = distance + (uphill / 1000)
print(f"Distance e: {round(distance_e2)}km-e")

## Durée

In [ ]:
print(f"{round(distance_e / KM_E_PER_DAY, 2)} jours")

## Altitude min & max

In [ ]:
if gpx.has_elevations():
  gpx.add_missing_elevations()
  (elevation_min, elevation_max) = gpx.get_elevation_extremes() # in meters
  print(f"Elevation max: {elevation_max}m / Elevation_min: {elevation_min}m")

## Modify metadata

In [ ]:
# gpx.description = "Un morceau du GR38, de Redon à Gourin"
# gpx.name = "GR38"
# gpx.author_email = "anne.lhote@gmail.com"
# gpx.author_name = "Anne L'Hôte"
# gpx.author_link = "https://github.com/annelhote"
# with open(gpx_file_name.replace(".gpx", "_extended.gpx"), "w") as f:
#   f.write(gpx.to_xml())